In [17]:
import nibabel as nib
import numpy as np
import os



coord_list = [(1,20,5),
              (-17,-4,5),
                (-21,-26,5),
              (17,-26,-1),]
coord_list_vox = [(27,40,29),
                  (18,28,29),
                    (16,17,29),
                    (35,17,26)
                  ]
input_path = r"C:\data\Networks\tmp_conversion\input"


reference_file_path = r"C:\github\dog_brain_toolkit\Atlas\Dog\Nitzsche\Czeibert_brain2mm.nii.gz"
reference_img = nib.load(reference_file_path)
reference_data = reference_img.get_fdata()



for indx, coord in enumerate(coord_list_vox):
    # Create an empty array with the same shape as the reference image
    input_data = np.zeros(reference_data.shape)
    x, y, z = coord
    input_data[x, y, z] = 20  # Set the voxel at the specified coordinate to 1
    # save the new image as a NIfTI file in the input directory
    input_img = nib.Nifti1Image(input_data, affine=reference_img.affine)
    input_file_path = os.path.join(input_path, f"{indx:02d}.nii.gz")
    nib.save(input_img, input_file_path)
    print(f"Saved: {input_file_path}")

Saved: C:\data\Networks\tmp_conversion\input\00.nii.gz
Saved: C:\data\Networks\tmp_conversion\input\01.nii.gz
Saved: C:\data\Networks\tmp_conversion\input\02.nii.gz
Saved: C:\data\Networks\tmp_conversion\input\03.nii.gz


In [12]:
z

312

In [20]:
output_path = r"C:\data\Networks\tmp_conversion\output"

for indx, coord in enumerate(coord_list):
    # Create an empty array with the same shape as the reference image
    output_file_path = os.path.join(output_path, f"{indx:02d}.nii.gz")
    output_img = nib.load(output_file_path)
    output_data = output_img.get_fdata()
    # find the coordinates of the maximum value in the output image
    max_coord = np.unravel_index(np.argmax(output_data, axis=None), output_data.shape)
    print(f"Input coord: {coord}, Output coord: {max_coord}")

Input coord: (1, 20, 5), Output coord: (17, 39, 41)
Input coord: (-17, -4, 5), Output coord: (9, 28, 41)
Input coord: (-21, -26, 5), Output coord: (7, 18, 41)
Input coord: (17, -26, -1), Output coord: (24, 18, 38)


In [2]:
# Step 4. Calculate rnd by repeating step 2 with permuted model
# input: pairwise similarity maps, permuted model
# output: rnd model similarity map for each subject, one per reps

# 18/Sep/2025
import os
import numpy as np
import nibabel as nib
import yaml
import sys
import time
from importlib import reload


method = 'correlation' # similarity method to use for searchlight
rsa_method = 'kendall'  # 'pearson','kendall','euclidean','mahalanobis', 'correlation'

'''
pearson: standard Pearson correlation
correlation: 1 - Pearson correlation (i.e., correlation distance)

'''

wait_time = 40 * 60  # 40 minutes
verbose = False
model = 'basic-block'
dataset = 'EmoB'
task = 'EmoB'
specie = 'D'
mask_type = 'b_GreyMatter2mm'
reps = 100

radius = 3
rsa_model = 'emotion-valence'

replace_file = False  # whether to overwrite existing output file

# Species label for atlas subfolder
if specie == 'D':
    specie_label = 'Dog'
    atlas_type = 'Nitzsche'  # force Nitzsche for dogs
elif specie == 'H':
    specie_label = 'Hum'
    atlas_type = 'MNI'  # force MNI for humans
else:
    raise ValueError(f"Unknown specie code: {specie}")



if os.name == 'nt':  # Windows
    datafolder = os.path.join(
        "P:\\userdata", 'raulh87', 'data'
    )
    git_folder = r"C:\github"
    
else:
    datafolder = os.path.join(
        '/home', 'raulh87', 'mnt', 'a471', 'userdata', 'raulh87', 'data'
    )
    #'/home/raulh87/mnt/a471/userdata/raulh87/github
    git_folder = os.path.join('/home', 'raulh87', 'mnt', 'a471', 'userdata', 'raulh87', 'github')

path_to_dog_brain_toolkit = os.path.join(git_folder, 'dog_brain_toolkit')
sys.path.append(path_to_dog_brain_toolkit)
import utils
reload(utils)
import preprocess_functions
reload(preprocess_functions)
import rsa_utils
reload(rsa_utils)
import utils_EmoB
reload(utils_EmoB)

mask = os.path.join(path_to_dog_brain_toolkit, 'Atlas', specie_label, atlas_type, mask_type + '.nii.gz')
rsa_model_path = datafolder + os.sep + dataset + os.sep + 'rsa_models' + os.sep + rsa_model + ".xlsx"
config_path = datafolder + os.sep + dataset + os.sep + 'config_files' + os.sep + model + '.yaml'
models_path = os.path.join(git_folder, 'dog_brain_toolkit', 'models')
# Load config.yaml
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
print(f"Loaded configuration from {config_path}")
# "P:\userdata\raulh87\data\EmoB\results\RSA\basic"

participants = config["participants"]
stim_types = config['stim_types']
# load the mask to use as reference
ref_img = nib.load(mask).get_fdata()
mask_affine = nib.load(mask).affine

project_dict = {
    "Dataset": config["dataset"],
    "Task": config["task"],
    "Participants": config["participants"],
    "Runs": config["runs"],
    "Sessions": config["sessions"],
    "Specie": config["specie"],
    "Atlas_type": config["atlas_type"],
    "Datafolder": datafolder,
}
results_path = datafolder + os.sep + dataset + os.sep + 'results'


for sub_N in participants:
    session_and_run_dict= utils_EmoB.get_session_and_run_list(specie, sub_N)
    for entry in session_and_run_dict:
        session = entry['session']
        run_N = entry['run']
        # correct session to 2 digits
        session = f"{session:02d}" 
        ## check if process was already done
        file_available, file_num_missing = preprocess_functions.check_file_status(
            project_dict, sub_N, run_N, session, process='model_similarity_maps_rnd',
            model=model, rsa_model=rsa_model, method=method, rsa_method=rsa_method, radius=radius,
            reps=reps, verbose=verbose
        )
        # file_available means that all files are done
        if file_available and not replace_file:
            if verbose:
                print(f"Existing sub-{sub_N:02d}, ses-{session}, run-{run_N:02d}, for {rsa_model} rnd. Skipping...")
            continue
        # make sure files needed are available
        file_available, _ = preprocess_functions.check_file_status(
            project_dict, sub_N, run_N, session, process='pairwise_similarity_maps',
            model=model, method=method, radius=radius, stim_types=stim_types)
        # if not available, skip or stop
        if not file_available:
            print(f"Missing for {specie}-sub-{sub_N:02d}, ses-{session}, run-{run_N:02d}, pairwise similarity maps. Skipping rnd model similarity computation.")
            continue
        
        # print
        print(f"sub-{sub_N:02d}, ses-{session}, run-{run_N:02d}, computing rnd model similarity maps...")
        # create temporal file to indicate that process is ongoing
        # "P:\userdata\raulh87\data\EmoB\results\RSA_rnd\basic-block\emotion-valence\D-sub-01\ses-01_task-EmoB_run-01_tmp.txt"
        temp_file = os.path.join(results_path, "RSA_rnd", model, rsa_model, f"{specie}-sub-{sub_N:02d}", f"ses-{session}_task-EmoB_run-{run_N:02d}_tmp.txt")
        # check if temp_file exists, if so, skip
        if os.path.exists(temp_file):
            # check if temporary file is older than wait_time
            if time() - os.path.getmtime(temp_file) < wait_time:
                print(f"Temporary file {temp_file} exists and is less than {wait_time/60} minutes old. Skipping computation.")
                continue
            else:
                print(f"Temporary file {temp_file} is older than {wait_time/60} minutes. Removing and reprocessing.")
                os.remove(temp_file)
        # if replace_file, then remove all existing rnd files
        if replace_file:
            rsa_utils.remove_existing_rnd_files(datafolder, dataset, sub_N, session, 
                                        run_N, specie, model, task, mask_type, 
                                        radius, rsa_model, method=method, 
                                        rsa_method=rsa_method, verbose=verbose, reps=reps)
        # check if 
        
        # create directory if it does not exist
        os.makedirs(os.path.dirname(temp_file), exist_ok=True)
        with open(temp_file, 'w') as f:
            f.write("Ongoing")
        print(f"Created temporary file: {temp_file}")
        try:
            # compute rnd model similarity maps
            # replace_file is set to false as existing files were already removed, will
            # only calculate missing files
            rsa_utils.compare_with_model(ref_img, mask_affine, datafolder, sub_N, session, 
                                            run_N, specie, model, dataset, task, mask_type, 
                                            radius, rsa_model, method=method, rsa_method=rsa_method, 
                                            replace_file=False, verbose=verbose, rnd=True, reps=reps)
        except Exception as e:
            print(f"Error computing rnd model similarity maps for sub-{sub_N:02d}, ses-{session}, run-{run_N:02d}: {e}")
            if os.path.exists(temp_file):
                os.remove(temp_file)
        # remove temporal file
        if os.path.exists(temp_file):
            os.remove(temp_file)


Loaded configuration from P:\userdata\raulh87\data\EmoB\config_files\basic-block.yaml
sub-01, ses-01, run-02, computing rnd model similarity maps...


TypeError: 'module' object is not callable

In [3]:
vals = range(1,10)
for i, val in enumerate(vals):
    print(f"Index: {i}, Value: {val}")



Index: 0, Value: 1
Index: 1, Value: 2
Index: 2, Value: 3
Index: 3, Value: 4
Index: 4, Value: 5
Index: 5, Value: 6
Index: 6, Value: 7
Index: 7, Value: 8
Index: 8, Value: 9


In [ ]:
# this script generates a basic model for beta maps and runs FSL FEAT
import shutil
import sys
import utils_EmoB
from importlib import reload
reload(utils_EmoB)

import os
import yaml
import os
import shutil
import argparse

import yaml
import getpass

sub_N = 10
model = 'basic'
dataset = 'EmoB'
redo_if_exists = False

userid = getpass.getuser()

# Determine data directory based on OS
if os.name == 'nt':  # Windows
    datafolder = os.path.join(
        "P:\\userdata", 'raulh87', 'data'
    )
    git_folder = r"C:\github"
    
else:
    datafolder = os.path.join(
        '/home', userid, 'mnt', 'a471', 'userdata', userid, 'data'
    )
    #'/home/raulh87/mnt/a471/userdata/raulh87/github
    git_folder = os.path.join('/home', userid, 'mnt', 'a471', 'userdata', userid, 'github')

models_path = os.path.join(git_folder, 'body_emo', 'models')
path_to_dog_brain_toolkit = os.path.join(git_folder, 'dog_brain_toolkit')
sys.path.append(path_to_dog_brain_toolkit)
import utils
reload(utils)
import preprocess_functions
reload(preprocess_functions)

config_path = datafolder + os.sep + dataset + os.sep + 'config_files' + os.sep + model + '.yaml'

# check if config_path exists
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file {config_path} not found.")


# Load config.yaml
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
print(f"Loaded configuration from {config_path}")


# GLM parameters
radius = config["radius"]
threshold = config["threshold"]
smooth = config["smooth"]
img_type = config["img_type"]
model = config["model"]
model_dict = config['model_dict']


base_design_file = model + '.fsf'


# Paths
dataset = config["dataset"]
task = config["task"]
specie = config["specie"]
run_list = config["runs"]
session_list = config["sessions"]
atlas_type = config['atlas_type']
img_type = config['img_type']
stim_types = config['stim_types']

design_template = path_to_dog_brain_toolkit + os.sep + 'FSL_designs' + os.sep + 'basic_DHRF.fsf'
design_template_modified = path_to_dog_brain_toolkit + os.sep + 'FSL_designs' + os.sep + 'basic_DHRF_modified.fsf'


    # Project configuration
project_dict = {
    "User": userid,
    "Dataset": config["dataset"],
    "Task": config["task"],
    "Participants": config["participants"],
    "Runs": config["runs"],
    "Sessions": config["sessions"],
    "Specie": config["specie"],
    "Atlas_type": config["atlas_type"],
    "Datafolder": datafolder,
}
# Get results path
results_path = utils.get_path('results', project_dict, local_data=False) 
results_path = os.path.join(results_path, 'GLM', model)
# get atlas file path
specie_label = 'Dog' if specie == 'D' else 'Hum'
atlas_file = os.path.join(
        path_to_dog_brain_toolkit, 'Atlas', specie_label, atlas_type, f"Czeibert_{img_type}.nii.gz"
    )
utils.generate_fsf(len(stim_types), design_template, design_template_modified)
# initialize labels to replace

# Loop over sessions and runs
for session in session_list:
    for run_N in run_list:
        # Check if GLM preprocessing is ready
        file_available, _ = preprocess_functions.check_file_status(
            project_dict, sub_N, run_N, session, process='GLM'
        )


        if not file_available:
            print(f"GLM preprocessing not ready for sub-{sub_N:02d}, ses-{session}, run-{run_N:02d}. Skipping...")
            continue
        
        # Output directory for FSL
        fsl_out = os.path.join(
            datafolder, dataset, 'results', 'GLM', model,
            f"{specie}-sub-{sub_N:02d}",
            f"ses-{session}_task-{task}_run-{run_N:02d}"
        )
        
        # Original and target movement file paths
        base_pre = os.path.join(
            datafolder, dataset, 'preprocessing',
            f"{specie}-sub-{sub_N:02d}",
            f"{specie}-sub-{sub_N:02d}_ses-{session}_task-{task}_run-{run_N:02d}.feat",
            'mc', 'prefiltered_func_data_mcf.par'
        )
        target_mov = os.path.join(
            datafolder, dataset, 'movement',
            f"{specie}-sub-{sub_N:02d}_ses-{session}_task-{task}_run-{run_N:02d}.par"
        )
        print(f"Copying movement file: {base_pre} -> {target_mov}")
        shutil.copyfile(base_pre, target_mov)
        mov_txt = os.path.join(
            datafolder, dataset, 'movement',
            f"{specie}-sub-{sub_N:02d}_ses-{session}_task-{task}_run-{run_N:02d}_fwd.txt"
        )
        print("Calculating framewise displacement...")
        preprocess_functions.fwd(
            base_pre, radius, threshold, output_file=mov_txt
        )
        # "P:\userdata\raulh87\data\EmoB\results\GLM\basic\D-sub-01\ses-01_task-EmoB_run-01.feat\rendered_thresh_zstat1.nii.gz"
        # check if zstat1 exists to determine if the GLM has been run
        zstat1_file = os.path.join(
            fsl_out + '.feat', 'rendered_thresh_zstat1.nii.gz'
        )
        if not redo_if_exists and os.path.exists(zstat1_file):
            print(f"Zstat1 file {zstat1_file} already exists. Skipping...")
            continue
        
        # # Check redo_if_exists
        # if not redo_if_exists and os.path.exists(fsl_out + '.feat'):
        #     # check if output .feat directory exists
        #     print(f"Output directory {fsl_out}.feat already exists. Skipping...")
        #     continue
        # Input NIfTI file
        input_nifti = os.path.join(
            datafolder, dataset, 'normalized',
            f"{specie}-sub-{sub_N:02d}",
            f"{specie}-sub-{sub_N:02d}_ses-{session}_task-{task}_run-{run_N:02d}.nii.gz"
        )
        # Atlas file
        atlas_file = os.path.join(
            path_to_dog_brain_toolkit, 'Atlas', specie_label, atlas_type, f"{img_type}.nii.gz"
        )
        # Extract TR and volumes
        TR, volumes = utils.extract_params(input_nifti)
        # Prepare FSF template replacement dictionary
        #design_in = os.path.join(datafolder, dataset, 'FSL_designs', base_design_file)
        design_out = os.path.join(datafolder, dataset, 'FSL_designs', model + f"{specie}-sub-{sub_N:02d}_tmp.fsf")
        labels = {
            'outputdir': (fsl_out,        'set fmri(outputdir)'),
            'TR':        (TR,             'set fmri(tr)'),
            'volumes':   (volumes,        'set fmri(npts)'),
            'BET':       (1 if specie=='H' else 0, 'set fmri(bet_yn)'),
            'smooth':    (smooth,         'set fmri(smooth)'),
            'input':     (input_nifti,    'set feat_files(1)'),
            'atlas':     (atlas_file,     'set fmri(regstandard)'),
            'movement':  (target_mov,     'set confoundev_files(1)'),
            #'condition': (cond_file,      'set fmri(custom1)'),
        }
        # add conditions based on stim_types
        for i, stim in enumerate(stim_types, start=1):
            # "C:\data\EmoB\models\all_types\D-sub-01\ses-01_task-EmoB_run-01\A.txt"
            cond_file = os.path.join(
                datafolder, dataset, 'models', model, f"{specie}-sub-{sub_N:02d}",
                f"ses-{session}_task-{task}_run-{run_N:02d}",
                f"{stim}.txt"
            )
            labels[f'condition{i}'] = (cond_file, f'set fmri(custom{i})')
     

        # Build replacement dict
        to_fill = {}
        for key, (val, find_str) in labels.items():
            rep = f'set {find_str.split()[1]}("{val}"' if isinstance(val, str) else f'set {find_str.split()[1]} {val}'
            # Actually ensure correct format
            if key in labels.keys():
                rep = f'{find_str} "{val}"'
            else:
                rep = f'{find_str} {val}'
            to_fill[key] = {
                'string_to_find': find_str,
                'string_to_replace': rep
            }
        # Fill FSF and run FSL
        utils.fill_fsf(to_fill, design_template_modified, design_out)
        # Remove existing .feat dir if present
        if os.path.exists(fsl_out + '.feat'):
            shutil.rmtree(fsl_out + '.feat')
        cmd = f'feat {design_out}'
        print(f"Running: {cmd}")
        if os.name != 'nt':
            os.system(cmd)


FileNotFoundError: Config file P:\userdata\raulh87\data\EmoB\config_files\basic.yaml not found.

: 

In [20]:
# This script computes similarity between two beta maps using a searchlight approach
# 5/Sep/2025
import os
import numpy as np
import nibabel as nib
import yaml
import sys
from importlib import reload
# map_1

sub_N = 8

model = 'basic'
dataset = 'EmoB'
task = 'EmoB'
specie = 'D'
mask_type = 'b_GreyMatter2mm'
atlas_type = 'Nitzsche'
radius = 3
method = 'pearson'  # 'pearson','kendall','euclidean','mahalanobis'
replace_file = False  # whether to overwrite existing output file
mask = r"C:\github\dog_brain_toolkit\Atlas\Dog\Nitzsche\b_GreyMatter2mm.nii.gz"

if os.name == 'nt':  # Windows
    datafolder = os.path.join(
        "P:\\userdata", 'raulh87', 'data'
    )
    git_folder = r"C:\github"
    
else:
    datafolder = os.path.join(
        '/home', userid, 'mnt', 'a471', 'userdata', userid, 'data'
    )
    #'/home/raulh87/mnt/a471/userdata/raulh87/github
    git_folder = os.path.join('/home', userid, 'mnt', 'a471', 'userdata', userid, 'github')


config_path = datafolder + os.sep + dataset + os.sep + 'config_files' + os.sep + model + '.yaml'
models_path = os.path.join(git_folder, 'dog_brain_toolkit', 'models')
# Load config.yaml
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

stim_types = config['stim_types']

path_to_dog_brain_toolkit = os.path.join(git_folder, 'dog_brain_toolkit')
sys.path.append(path_to_dog_brain_toolkit)
import utils
reload(utils)
import preprocess_functions
reload(preprocess_functions)


stim_types = config['stim_types']



run_list = config["runs"]
session_list = config["sessions"]

project_dict = {
    "Dataset": config["dataset"],
    "Task": config["task"],
    "Participants": config["participants"],
    "Runs": config["runs"],
    "Sessions": config["sessions"],
    "Specie": config["specie"],
    "Atlas_type": config["atlas_type"],
    "Datafolder": datafolder,
}


for session in session_list:
    for run_N in run_list:
        # Check if GLM preprocessing is ready
        file_available, filename = preprocess_functions.check_file_status(
            project_dict, sub_N, run_N, session, process='GLM'
        )
        if not file_available:
            # print message and skip
            print(f"GLM file: {filename} not available. Skipping...")
            continue
        

        for stim_N1 in range(len(stim_types)):
            stim_1_name = stim_types[stim_N1]
            file_1_path = datafolder + os.sep + dataset + os.sep + 'results' + os.sep + 'GLM' + os.sep + model + os.sep + f"{specie}-sub-{sub_N:02d}" + os.sep + f"ses-{session}_task-{task}_run-{run_N:02d}.feat" + os.sep + 'stats' + os.sep + f"pe{stim_N1+1}.nii.gz"
            print(f"Processing similarity for stim {stim_1_name}, file: {file_1_path}")
            for stim_N2 in range(stim_N1 + 1, len(stim_types)):
                # if stim_N1 == stim_N2, skip
                if stim_N1 == stim_N2:
                    print("Skipping identical stimuli...")
                    continue
                file_2_path = datafolder + os.sep + dataset + os.sep + 'results' + os.sep + 'GLM' + os.sep + model + os.sep + f"{specie}-sub-{sub_N:02d}" + os.sep + f"ses-{session}_task-{task}_run-{run_N:02d}.feat" + os.sep + 'stats' + os.sep + f"pe{stim_N2+1}.nii.gz"

                print(f"  Comparing with stim {stim_N2}, file: {file_2_path}")

                stim_2_name = stim_types[stim_N2]

                # file_1_path = r"P:\userdata\raulh87\data\EmoB\results\GLM\basic\D-sub-01\ses-01_task-EmoB_run-01.feat\stats\pe1.nii.gz"
                
                
                output_path = datafolder + os.sep + dataset + os.sep + 'results' + os.sep + 'RSA' + os.sep + model + os.sep + f"{specie}-sub-{sub_N:02d}" + os.sep + f"ses-{session}_task-{task}_run-{run_N:02d}" + os.sep + f"r-{radius}_{method}_{stim_N1}_{stim_N2}.nii.gz"

                # check if output_path exists
                if os.path.exists(output_path) and not replace_file:
                    print(f"Output file {output_path} already exists. Skipping...")
                    continue

                file_1_map = nib.load(file_1_path).get_fdata()
                file_2_map = nib.load(file_2_path).get_fdata()

                similarity_map = utils.similarity_searchlight(file_1_map, file_2_map, nib.load(mask).get_fdata().astype(bool), radius=radius, method=method)
                # check if output directory exists
                if not os.path.exists(os.path.dirname(output_path)):
                    os.makedirs(os.path.dirname(output_path), exist_ok=True)
                nib.save(nib.Nifti1Image(similarity_map, nib.load(file_1_path).affine), output_path)
                print(f"Saved similarity map to {output_path}")



FileNotFoundError: [Errno 2] No such file or directory: 'P:\\userdata\\raulh87\\data\\EmoB\\config_files\\basic.yaml'

In [12]:
session

'01'

In [11]:
datafolder + os.sep + dataset + os.sep + 'results' + os.sep + 'GLM' + os.sep + model + os.sep + f"{specie}-sub-{sub_N:02d}" + os.sep + f"ses-{session:02d}_task-{task}_run-{run_N:02d}.feat" + os.sep + 'stats' + os.sep + f"pe{stim_N1+1}.nii.gz"

ValueError: Unknown format code 'd' for object of type 'str'

In [8]:
sessio

'01'

In [ ]:
# Step 1 v7. added missing log 26/Feb/2025 This script will read the video files overview.xlsx file and extract the properties of the videos
# input: videos, video files overview.xlsx
# output: pkl files with the properties of the videos stored in a dictionary for each video
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import utils
from importlib import reload
reload(utils)

username = 'raulh87'


# if system is windows
if os.name == 'nt':
    # props_folder = r"P:\userdata\raulh87\props"
    props_folder = r"C:\github\visual_comp\props"
    table_path = r"C:\github\visual_comp\video files overview.xlsx"
else:
    props_folder = '/home' + os.sep + username + '/mnt/a471/userdata/' + username + 'videos' + os.sep + 'props'
    table_path = '/home' + os.sep + username + '/mnt/a471/userdata/' + username + 'videos' + os.sep + 'video files overview.xlsx'

missing_log = []
missing_log_error = []
missing_log_no_time = []
table = pd.read_excel(table_path)
for indx,row in table.iterrows():
    

    if row['keep'] == 'n': # skip if keep is n
        continue
    date_string = row['date']
    date_string = date_string.strftime('%d%b%Y')
    performer = row['participant_name']
    performer_id = utils.get_performer_id(performer)
    scene_specification = row['scene_specification']
    scene_emotion = row['scene_emotion']
    scene_type = utils.get_scene_type(scene_emotion)
    video_counter = row['video_counter']
    primary_key = f"{scene_type}_{performer_id}_{video_counter:05d}"


    if os.name == 'nt':
        videos_folder = r"P:\userdata\raulh87\videos" + os.sep + date_string + os.sep + performer
    else:
        videos_folder =  '/home' + os.sep + username + '/mnt/a471/userdata/' + username + '/videos' + os.sep + date_string + os.sep + performer
    start = row['time window start']
    end = row['time window end']
    if pd.isnull(start) or pd.isnull(end):
        print(f"No time window for {primary_key}")
        missing_log_no_time.append(primary_key)
        continue

    video_path = os.path.join(videos_folder, row['file name'] + '.mp4')

    # check if a file with the props have already been written, if not run save_video_props
    if os.path.exists(os.path.join(props_folder, primary_key + '.pkl')):
        continue
        # try to load it
        try:
            with open(os.path.join(props_folder, primary_key + '.pkl'), 'rb') as f:
                tmp = pickle.load(f)
            print(f"Video properties already saved for {primary_key}")
            continue

        except:
            # if it fails, delete the file and add video_path to missing_log
            # os.remove(os.path.join(props_folder, primary_key + '.pkl'))
            # print(f"Error loading file for {primary_key}.pkl, file deleted")
            print(f"Error loading file for {primary_key}.pkl")

    # check if file exists, if not skip
    if not os.path.exists(video_path):
        print(f"File not found: {video_path}")
        missing_log.append(video_path)
        continue

    
        
    else:
        # create an empty file primary_key.pkl
        open(os.path.join(props_folder, primary_key + '.pkl'), 'a').close()
        # indicate that an empty file has been created
        print(f"Empty file created for {primary_key}")
        try:
            utils.save_video_props(video_path, props_folder, primary_key, start, end)
            # indicate that the file has been processed
            print(f"Video properties saved for {primary_key}")
        except:
            os.remove(os.path.join(props_folder, primary_key + '.pkl'))
            # add video_path to missing_log
            missing_log_error.append(video_path)

print("Finished processing all videos")
print("Missing videos:")
for video in missing_log:
    print(video)
print("Error processing videos:")
for video in missing_log_error:
    print(video)
print("Videos with no time window:")
for video in missing_log_no_time:
    print(video)

No time window for HA_TH_00141
No time window for AN_TH_00147
No time window for FE_TH_00151
No time window for SU_TH_00152
No time window for SU_TH_00153
No time window for SU_TH_00154
No time window for SU_TH_00155
No time window for DI_TH_00156
No time window for DI_TH_00157
No time window for DI_TH_00158
No time window for SA_TH_00159
No time window for SA_TH_00160
No time window for HA_TH_00161
No time window for HA_TH_00162
No time window for HA_TH_00163
No time window for HA_TH_00164
No time window for HA_TH_00165
No time window for SA_TH_00166
No time window for SA_TH_00167
No time window for SA_TH_00168
No time window for SA_TH_00169
No time window for AN_TH_00170
No time window for AN_TH_00171
No time window for AN_TH_00172
No time window for AN_TH_00173
No time window for AN_TH_00174
No time window for FE_TH_00175
No time window for FE_TH_00176
No time window for FE_TH_00177
No time window for FE_TH_00178
No time window for SU_TH_00179
No time window for SU_TH_00180
No time 

: 

In [ ]:
# Step 2. This script will read the video files overview.xlsx file and extract the properties of the videos

import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os


username = 'raulh87'
props_folder = r"C:\github\visual_comp\props"

# if system is windows

table_path = r"C:\github\visual_comp\video files overview.xlsx"
table = pd.read_excel(table_path)
for indx,row in table.iterrows():
    if row['keep'] == 'n': # skip if keep is n
        continue
    date_string = row['date']
    date_string = date_string.strftime('%d%b%Y')
    performer = row['participant_name']
    performer_id = utils.get_performer_id(performer)
    scene_specification = row['scene_specification']
    scene_emotion = row['scene_emotion']
    scene_type = utils.get_scene_type(scene_emotion)
    video_counter = row['video_counter']
    primary_key = f"{scene_type}_{performer_id}_{video_counter:05d}"


    if os.name == 'nt':
        videos_folder = r"P:\userdata\raulh87\videos" + os.sep + date_string + os.sep + performer
    else:
        videos_folder =  '/home' + os.sep + username + '/mnt/a471/userdata/' + username + '/videos' + os.sep + date_string + os.sep + performer
    start = row['time window start']
    end = row['time window end']

    video_path = os.path.join(videos_folder, row['file name'] + '.mp4')
    print('analyzing video: ', video_path)

    # check if file exists, if not skip
    if not os.path.exists(video_path):
        print(f"File not found: {video_path}")
        continue

    # print each value
    print('primary key: ', primary_key)
    
    # section_name based on performer_id, 5 zero padded primary key, 
    


    
    # check if a file with the props have already been written, if not run save_video_props
    if os.path.exists(os.path.join(props_folder, primary_key + '.pkl')):
        print(f"Video properties already saved for {primary_key}")
        # skip the video
        continue
        
    else:
        # indicate that the file will be processed
        print(f"Processing video {primary_key}")
        # try running save_video_props if it fails erase the file corresponding to primary_key
        # create an empty file primary_key.pkl
        open(os.path.join(props_folder, primary_key + '.pkl'), 'a').close()

        try:
            utils.save_video_props(video_path, props_folder, primary_key, start, end)
        except:
            os.remove(os.path.join(props_folder, primary_key + '.pkl'))
            
        # save_video_props(video_path, props_folder, primary_key, start, end)
    
    

In [ ]:
# Step 2. This script will read the video files overview.xlsx file and extract the properties of the videos

# This script will plot the properties of the videos usind mds

import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import utils
from importlib import reload
reload(utils)


username = 'raulh87'
props_folder = r"C:\github\visual_comp\props"

# create res_table
res_table = pd.DataFrame()


table_path = r"C:\github\visual_comp\video files overview.xlsx"
table = pd.read_excel(table_path)
# add primary_key to the table
table['primary_key'] = ''

counter_loaded, counter_not_loaded = 0, 0 # initialize counters

for indx,row in table.iterrows():
    if row['keep'] == 'n': # skip if keep is n
        continue
    date_string = row['date']
    date_string = date_string.strftime('%d%b%Y')
    performer = row['participant_name']
    performer_id = utils.get_performer_id(performer)
    scene_specification = row['scene_specification']
    scene_emotion = row['scene_emotion']
    scene_type = utils.get_scene_type(scene_emotion)
    # primary_key = row['primary_key']
    video_counter = row['video_counter']
    primary_key = f"{scene_type}_{performer_id}_{video_counter:05d}"
    # add primary_key to the table
    # table.at[indx, 'primary_key'] = primary_key
    # try to load the video properties
    try:
        # load the video properties saved as props_folder/{}
        with open(os.path.join(props_folder, primary_key + '.pkl'), 'rb') as f:
            video_props = pickle.load(f)
            print(f"Loaded {os.path.join(props_folder, primary_key + '.pkl')}")
            f.close()
            # increment the counter
            counter_loaded += 1
    except:
        # print the full path
        print(f"Error loading {os.path.join(props_folder, primary_key + '.pkl')}")
        # increment the counter
        counter_not_loaded += 1
        # skip the video
        continue
    # calculate average hue, saturation, brightness, contrast, motion
    avg_hue = sum(video_props['hue']) / len(video_props['hue'])
    avg_saturation = sum(video_props['saturation']) / len(video_props['saturation'])
    avg_brightness = sum(video_props['brightness']) / len(video_props['brightness'])
    avg_contrast = sum(video_props['contrast']) / len(video_props['contrast'])
    avg_motion = sum(video_props['motion']) / len(video_props['motion'])
    # create a row with the properties
    new_row = {
        'primary_key': primary_key,
        'avg_hue': avg_hue,
        'avg_saturation': avg_saturation,
        'avg_brightness': avg_brightness,
        'avg_contrast': avg_contrast,
        'avg_motion': avg_motion,
    }
    # concatenate the row to the res_table
    res_table = pd.concat([res_table, pd.DataFrame([new_row])])

# print the number of videos loaded and not loaded
print(f"Videos loaded: {counter_loaded}, Videos not loaded: {counter_not_loaded}")

    

Loaded C:\github\visual_comp\props\TS_BE_00001.pkl
Loaded C:\github\visual_comp\props\TS_BE_00002.pkl
Loaded C:\github\visual_comp\props\CL_BE_00003.pkl
Loaded C:\github\visual_comp\props\CL_BE_00004.pkl
Loaded C:\github\visual_comp\props\LH_BE_00005.pkl
Loaded C:\github\visual_comp\props\LH_BE_00006.pkl
Loaded C:\github\visual_comp\props\HA_BE_00007.pkl
Loaded C:\github\visual_comp\props\HA_BE_00008.pkl
Loaded C:\github\visual_comp\props\HA_BE_00009.pkl
Loaded C:\github\visual_comp\props\AN_BE_00010.pkl
Loaded C:\github\visual_comp\props\AN_BE_00011.pkl
Loaded C:\github\visual_comp\props\AN_BE_00012.pkl
Loaded C:\github\visual_comp\props\AN_BE_00013.pkl
Loaded C:\github\visual_comp\props\AN_BE_00014.pkl
Loaded C:\github\visual_comp\props\SA_BE_00015.pkl
Loaded C:\github\visual_comp\props\SA_BE_00016.pkl
Loaded C:\github\visual_comp\props\SA_BE_00017.pkl
Loaded C:\github\visual_comp\props\SA_BE_00018.pkl
Loaded C:\github\visual_comp\props\FE_BE_00019.pkl
Loaded C:\github\visual_comp\pr

In [ ]:
res_table

,primary_key,avg_hue,avg_saturation,avg_brightness,avg_contrast,avg_motion
0,TS_BE_00001,42.972537,37.796145,116.759083,51.058081,0.295814
0,TS_BE_00002,43.161206,39.810079,115.727588,51.611605,0.467928
0,CL_BE_00003,40.491364,35.187362,116.409921,50.510035,0.205433
0,CL_BE_00004,40.594126,35.028860,116.466242,50.569694,0.199238
0,LH_BE_00005,39.628413,36.133832,115.734998,50.051174,0.444738
...,...,...,...,...,...,...
0,SA_KA_00319,53.144968,34.875104,104.842427,42.608544,0.190767
0,SA_KA_00320,53.300715,35.173708,104.840831,42.835614,0.193406
0,SA_KA_00321,48.005788,37.761975,106.330516,44.402726,0.179249
0,SA_KA_00322,52.135421,34.679875,104.721000,42.450108,0.194050


In [6]:
table

,file name,keep,video_counter,participant_key,participant_name,m/f,date,scene code,scene_emotion,scene_specification,time window start,time window end,notes,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,Gh011042,n,0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,useful for calibration of camera frame?,NaN,NaN,NaN,NaN
1,Gh011043,y,1,1.0,Bettina,f,2024-12-06,0a,two small steps forward,control,00:13:00,00:18:00,NaN,NaN,NaN,1.0,1.0
2,Gh011043,y,2,1.0,Bettina,f,2024-12-06,0a,two small steps forward,control,00:21:00,00:25:00,NaN,NaN,NaN,NaN,NaN
3,Gh011043,y,3,1.0,Bettina,f,2024-12-06,0b,changing light bulb,control,01:07:00,01:16:00,NaN,NaN,NaN,NaN,NaN
4,Gh011043,y,4,1.0,Bettina,f,2024-12-06,0b,changing light bulb,control,01:09:00,01:17:00,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721,Gh011408,NaN,686,13.0,Peter,m,2025-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
722,Gh011409,NaN,686,13.0,Peter,m,2025-02-18,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
723,Gh011410,y,687,13.0,Peter,m,2025-02-18,1c,anger,human w cue,00:11:00,00:28:00,NaN,NaN,NaN,NaN,NaN
724,Gh011411,NaN,687,13.0,Peter,m,2025-02-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Step 1. V3 This script will read the video files overview.xlsx file and extract the properties of the videos

import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os


username = 'raulh87'
props_folder = r"C:\github\visual_comp\props"

# if system is windows

table_path = r"C:\github\visual_comp\video files overview.xlsx"
table = pd.read_excel(table_path)
for indx,row in table.iterrows():
    if row['keep'] == 'n': # skip if keep is n
        continue
    date_string = row['date']
    date_string = date_string.strftime('%d%b%Y')
    performer = row['participant_name']
    performer_id = get_performer_id(performer)
    scene_specification = row['scene_specification']
    scene_emotion = row['scene_emotion']
    scene_type = get_scene_type(scene_emotion)
    video_counter = row['video_counter']
    primary_key = f"{scene_type}_{performer_id}_{video_counter:05d}"


    if os.name == 'nt':
        videos_folder = r"P:\userdata\raulh87\videos" + os.sep + date_string + os.sep + performer
    else:
        videos_folder =  '/home' + os.sep + username + '/mnt/a471/userdata/' + username + '/videos' + os.sep + date_string + os.sep + performer
    start = row['time window start']
    end = row['time window end']

    video_path = os.path.join(videos_folder, row['file name'] + '.mp4')
    print('analyzing video: ', video_path)

    # check if file exists, if not skip
    if not os.path.exists(video_path):
        print(f"File not found: {video_path}")
        continue

    # print each value
    print('primary key: ', primary_key)
    
    # section_name based on performer_id, 5 zero padded primary key, 
    


    
    # check if a file with the props have already been written, if not run save_video_props
    if os.path.exists(os.path.join(props_folder, primary_key + '.pkl')):
        print(f"Video properties already saved for {primary_key}")
        # skip the video
        continue
        
    else:
        # indicate that the file will be processed
        print(f"Processing video {primary_key}")
        # try running save_video_props if it fails erase the file corresponding to primary_key
        # create an empty file primary_key.pkl
        open(os.path.join(props_folder, primary_key + '.pkl'), 'a').close()
        # indicate that an empty file has been created
        print(f"Empty file created for {primary_key}")
        try:
            utils.save_video_props(video_path, props_folder, primary_key, start, end)
        except:
            os.remove(os.path.join(props_folder, primary_key + '.pkl'))
            
        # save_video_props(video_path, props_folder, primary_key, start, end)
    
    

analyzing video:  P:\userdata\raulh87\videos\06Dec2024\Bettina\Gh011043.mp4
primary key:  TS_BE_00001
Video properties already saved for TS_BE_00001
analyzing video:  P:\userdata\raulh87\videos\06Dec2024\Bettina\Gh011043.mp4
primary key:  TS_BE_00002
Video properties already saved for TS_BE_00002
analyzing video:  P:\userdata\raulh87\videos\06Dec2024\Bettina\Gh011043.mp4
primary key:  CL_BE_00003
Processing video CL_BE_00003
Start is:   67
Processing frame 1960.0 of 3999.0
Processing frame 1980.0 of 3999.0
Processing frame 2000.0 of 3999.0


FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\github\\visual_comp\\props\\CL_BE_00003.pkl'

In [16]:
res_table

,primary_key,avg_hue,avg_saturation,avg_brightness,avg_contrast,avg_motion
0,TS_BE_00001,42.972537,37.796145,116.759083,51.058081,0.295814
0,TS_BE_00002,43.161206,39.810079,115.727588,51.611605,0.467928
0,SU_KI_00079,31.444327,32.597487,120.992553,48.567494,0.212879
0,SU_KI_00080,30.657682,31.125238,122.749549,48.367888,0.192614
0,TS_AM_00081,34.710124,30.637549,118.752437,49.439405,0.216885
0,TS_AM_00082,35.104011,31.420418,117.284263,49.724990,0.205614
0,HA_AM_00083,35.024278,32.680825,117.648081,51.061394,0.377442


In [7]:
video_props.keys()

dict_keys(['frame_count', 'fps', 'hue', 'saturation', 'brightness', 'contrast', 'motion'])

In [ ]:
video_counter